In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as cp
from scipy.interpolate import griddata
from matplotlib.colors import LogNorm
from matplotlib.gridspec import GridSpec
from scipy.interpolate import interp1d
from FallbackGen import FallbackGen
from TDECalculator import TDECalculator
import gc

In [2]:
MBH = 1e6
Rp = 25
a = 0.2
N = 5000
E = 1.0
Q = 0.0
orbit="rel"

In [3]:
mass_p = TDECalculator('MAMS1Msun', orbit, MBH, Rp, a, N=N)

In [4]:
sample_p = mass_p.rel_whole_star_sample()

In [5]:
radii_p = sample_p['rr']

rtde = mass_p.R_TDE
Lz = mass_p.mom_kerr_analytic(Rp, a)
E = 1.0
Q = 0

radii = np.where(
    radii_p <= 0.5,
    rtde - radii_p * mass_p.Rstar,
    rtde + radii_p * mass_p.Rstar
)

deltaE = mass_p.Rstar / mass_p.Rp**2 

i = int(np.random.uniform(0, 1132))
j = int(np.random.uniform(0, 90000))
dE = sample_p['dEnergy_random'] * deltaE 
dLz = mass_p.dLz_random
dQ = mass_p.dQ_random
mass_ratio = mass_p.mass_ratio

In [6]:
print(f"Lz = {Lz}")
print(f"dLz range: [{dLz.min():.4e}, {dLz.max():.4e}]")
print(f"total_Lz range: [{(Lz + dLz).min():.4e}, {(Lz + dLz).max():.4e}]")
print(f"dE range: [{dE.min():.4e}, {dE.max():.4e}]")
print(f"total_E range: [{(E + dE).min():.4e}, {(E + dE).max():.4e}]")
print(f"bound fraction: {((E + dE) < 1.0).mean():.3f}")

Lz = 7.354962919731006
dLz range: [-8.2945e+00, 8.2960e+00]
total_Lz range: [-9.3957e-01, 1.5651e+01]
dE range: [-4.7169e-02, 4.7174e-02]
total_E range: [9.5283e-01, 1.0472e+00]
bound fraction: 0.502


In [7]:
dT_p = FallbackGen(mass_ratio, a, radii, Rp, E, Lz, Q, dE, dLz, dQ, N)
np.save('dT_p.npy', dT_p.dTs)
gc.collect()

Computing radial periods for 116,280,000 particles ...
  E  range: [0.952831, 1.047174]
  Q  range: [-4.916e-03, 4.917e-03]
  chunk_size = 50,000
  Finding roots (chunked eigensolver) ...
  roots chunk 11667/11667 (100%)
  Bound: 58,331,526 / 116,280,000
  Valid roots: 57,834,380
  Valid Lambda_r: 57,834,380
  Quadrature: 1157 chunks ...
    chunk 1157/1157  (100%)
  Successful T_r: 57,834,380 / 116,280,000


20

In [8]:
def make_plot_dicts(whole_star_sample, dT, delta):
    dE_rand = whole_star_sample['dEnergy_random']
    dT_rand = dT / delta
    dMass   = whole_star_sample['dMass']

    bins_E = np.linspace(-2, 2, 1000)
    bins_T = np.logspace(0, 6, 1000)

    # dE: use all particles with finite energy and mass
    valid_E = np.isfinite(dE_rand)
    hist_E, edges_E = np.histogram(dE_rand[valid_E], bins=bins_E,
                                   weights=dMass[valid_E], density=True)

    # dT: only particles with valid finite period within bin range
    valid_T = (np.isfinite(dT_rand) & np.isfinite(dE_rand)
               & (dT_rand >= 1.0) & (dT_rand <= 1e6))
    hist_T, edges_T = np.histogram(dT_rand[valid_T], bins=bins_T,
                                   weights=dMass[valid_T], density=True)

    return {"x": 0.5*(edges_E[:-1]+edges_E[1:]), "y": hist_E}, \
           {"x": 0.5*(edges_T[:-1]+edges_T[1:]), "y": hist_T}

In [9]:
DeltaE = mass_p.Rstar / mass_p.Rp**2
DeltaT = 1 / DeltaE**1.5

In [10]:
rel_p_E, rel_p_T = make_plot_dicts(sample_p, dT_p.dTs, DeltaT)

In [11]:
import json

adden = "m2_rp25_a0p2"

with open(f"Fallback_Data/rel_p_e_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_p_E.items()}, f)
with open(f"Fallback_Data/rel_p_t_{adden}.txt", "w") as f:
    json.dump({k: v.tolist() for k, v in rel_p_T.items()}, f)